In [2]:
# Flatten CSVs from subfolders into one output folder, renaming each CSV
# to its parent folder name. Copy exactly one readme.txt into the output.

from pathlib import Path
import shutil
import hashlib

# ==== EDIT ME ====
ROOT_DIR   = Path("./VIIRS_12-24")          # folder that contains the subfolders
OUTPUT_DIR = Path("./collected_csvs")# output folder to create/fill
OVERWRITE  = False                          # set True to overwrite existing files
# =================

def _sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def collect_csvs(root_dir: Path, output_dir: Path, overwrite: bool = False):
    if not root_dir.is_dir():
        raise FileNotFoundError(f"Root directory not found: {root_dir}")

    output_dir.mkdir(parents=True, exist_ok=True)

    kept_readme_path = None
    kept_readme_hash = None
    copied = 0
    skipped = 0

    # Only immediate subfolders of root_dir
    for sub in sorted(p for p in root_dir.iterdir() if p.is_dir()):
        # 1) CSV: take the first *.csv in the subfolder (if multiple, warn & use first)
        csvs = sorted(sub.glob("*.csv"))
        if not csvs:
            print(f"[skip] No CSV found in: {sub}")
            continue
        if len(csvs) > 1:
            print(f"[note] Multiple CSVs in {sub}; using {csvs[0].name}")

        dst_csv = output_dir / f"{sub.name}.csv"
        if dst_csv.exists() and not overwrite:
            print(f"[skip] {dst_csv.name} already exists; set OVERWRITE=True to replace.")
            skipped += 1
        else:
            shutil.copy2(csvs[0], dst_csv)
            copied += 1

        # 2) README: keep only the first readme.txt encountered (case-insensitive)
        readmes = [p for p in sub.iterdir() if p.is_file() and p.name.lower() == "readme.txt"]
        if readmes:
            if kept_readme_path is None:
                kept_readme_path = output_dir / "readme.txt"
                shutil.copy2(readmes[0], kept_readme_path)
                kept_readme_hash = _sha256(readmes[0])
            else:
                # Warn if a later README differs
                if _sha256(readmes[0]) != kept_readme_hash:
                    print(f"[note] README in {sub} differs; keeping the first one already copied.")

    print(f"\nDone. Copied {copied} CSV(s); skipped {skipped}.")
    if kept_readme_path:
        print(f"Kept README at: {kept_readme_path}")
    else:
        print("No readme.txt found; none copied.")

# Run it
collect_csvs(ROOT_DIR, OUTPUT_DIR, overwrite=OVERWRITE)



Done. Copied 13 CSV(s); skipped 0.
Kept README at: collected_csvs/readme.txt
